# PQQA vs SA vs PA — Colab walkthrough

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yuma-Ichikawa/QQA4CO/blob/main/examples/15_pqqa_sa_pa.ipynb)

This notebook walks through the three core annealing backends shipped with **QQA4CO** and shows what each one is *uniquely* good at:

| Backend | What it returns | Best for |
|---|---|---|
| **PQQA** (`qqa.anneal`) | Lowest-energy configuration (gradient-based, parallel) | Wall-clock speed on hard combinatorial problems |
| **SA** (`qqa.simulated_annealing`) | Lowest-energy configuration (single-spin MCMC) | Textbook baseline |
| **PA** (`qqa.population_annealing`) | Lowest-energy configuration **+ equilibrium sample at β_end + free-energy estimate + family tree** | Estimating thermodynamic observables, free energies, and inspecting how the population collapses |

Sections:

1. Install and imports
2. Define a problem (Sherrington–Kirkpatrick spin glass on N = 30)
3. Run **PQQA**, **SA**, and **PA** on the same problem
4. Plot the convergence trace of all three
5. **PA-only**: equilibrium sample histogram at β_end
6. **PA-only**: free-energy density estimate vs β (Hukushima–Iba) and a closed-form sanity check on a 1-spin Ising
7. **PA-only**: resampling **family tree** — which initial replicas survived, and when did the population collapse?

> **Resampling correction is implicit.** PA's free-energy estimator uses the *unnormalised* reweighting factor averaged over the current population at every annealing step (Hukushima & Iba, 2003). The resampling step preserves the average weight in expectation, which is precisely why this works.

## 1. Install QQA4CO

Uncomment the `pip install` line on Colab. Locally, install with `uv sync` from the repo root.

In [ ]:
# !pip install -q qqa  # uncomment on Colab
import math

import matplotlib.pyplot as plt
import numpy as np
import torch

import qqa

qqa.fix_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"qqa {qqa.__version__}, torch {torch.__version__}, device={device}")

## 2. Define a problem — Sherrington–Kirkpatrick spin glass

We use a small SK instance (N = 30, seed = 0) so each backend finishes in seconds even on CPU. PQQA / SA / PA all consume the same `qqa.SherringtonKirkpatrick` problem object.

In [ ]:
N = 30
problem = qqa.SherringtonKirkpatrick(N=N, seed=0)
print(f"problem = {type(problem).__name__}, num_vars = {problem.num_spins}")

## 3. Run all three backends at a matched compute budget

We pick:
* PQQA: `sol_size=128`, `num_epochs=600`
* SA: `sol_size=128`, `num_sweeps=600` (matches PQQA epochs ≈ MCMC sweeps)
* PA: `sol_size=128`, `num_temps=60`, `sweeps_per_temp=10` (so `T*K = 600` ≈ SA's sweep count)

All three share `beta_end = 8.0` so PA's free-energy estimate is comparable across runs.

In [ ]:
BETA_END = 8.0
BETA_START = 0.05

res_pqqa = qqa.anneal(
    problem,
    sol_size=128,
    num_epochs=600,
    min_bg=-3.0,
    max_bg=0.1,
    device=device,
    verbose=False,
)
print(f"PQQA best={res_pqqa.best_obj:.4f}  runtime={res_pqqa.runtime:.2f}s")

In [ ]:
res_sa = qqa.simulated_annealing(
    problem,
    sol_size=128,
    num_sweeps=600,
    beta_start=BETA_START,
    beta_end=BETA_END,
    device=device,
    verbose=False,
)
print(f"SA   best={res_sa.best_obj:.4f}  runtime={res_sa.runtime:.2f}s")

In [ ]:
res_pa = qqa.population_annealing(
    problem,
    sol_size=128,
    num_temps=60,
    sweeps_per_temp=10,
    beta_start=BETA_START,
    beta_end=BETA_END,
    resample="systematic",
    record_genealogy=True,
    device=device,
    verbose=False,
)
print(
    f"PA   best={res_pa.best_obj:.4f}  runtime={res_pa.runtime:.2f}s  "
    f"F/N={res_pa.free_energy_density:.4f}  ln Z={res_pa.log_z:.3f}"
)

## 4. Convergence trace of all three

PA's x-axis is the temperature step, PQQA's is the epoch, and SA's is the sweep. They have different units of "work", but the curves answer the question "how quickly did each backend reach its best objective?"

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5), facecolor="white")
ax.plot(res_pqqa.history["best_obj"], label="PQQA", color="#0050C8", lw=2)
ax.plot(res_sa.history["best_obj"], label="SA", color="#8c8c8c", lw=2, ls="--")
ax.plot(res_pa.history["best_obj"], label="PA", color="#C8005A", lw=2, ls=":")
ax.set_xlabel("iteration (PQQA epoch / SA sweep / PA temperature step)")
ax.set_ylabel("best loss so far")
ax.set_title("PQQA vs SA vs PA — convergence")
ax.legend()
ax.grid(ls="--", alpha=0.5)
plt.show()

## 5. PA equilibrium sample at β_end

After the final round of MCMC sweeps, PA's `final_x` holds the **equilibrium population** at `β_end`. These are approximately Boltzmann samples at the target inverse temperature — useful for estimating physical observables (magnetisation, energy distribution, susceptibility, ...). PQQA / SA only return the single best configuration.

In [ ]:
final_e = res_pa.final_loss.cpu().numpy()
print(
    f"final population: R={len(final_e)}\n"
    f"  mean E={final_e.mean():.4f}\n"
    f"  min  E={final_e.min():.4f}\n"
    f"  std  E={final_e.std():.4f}"
)

fig, ax = plt.subplots(figsize=(7, 3.5), facecolor="white")
ax.hist(final_e, bins=24, color="#C8005A", alpha=0.85)
ax.axvline(res_pa.best_obj, color="#222", ls="--", lw=1.4, label=f"best = {res_pa.best_obj:.3f}")
ax.axvline(final_e.mean(), color="#0050C8", ls=":", lw=1.4, label=f"mean = {final_e.mean():.3f}")
ax.set_xlabel("Energy at β_end")
ax.set_ylabel("Replicas")
ax.set_title("PA equilibrium population — energy histogram at β_end")
ax.legend()
ax.grid(ls="--", alpha=0.4)
plt.show()

## 6. PA free-energy density vs β

PA tracks the partition function via the Hukushima–Iba estimator:

$$\ln Z(\beta_t) - \ln Z(\beta_{t-1}) \;=\; \ln \left( \frac{1}{R} \sum_{r=1}^{R} e^{-\Delta\beta_t \cdot E_r^{(t)}} \right)$$

where $E_r^{(t)}$ is the energy of replica $r$ **just before** the resampling at step $t$. The resampling correction is implicit because we average the *unnormalised* weights — see Hukushima & Iba (2003) and Machta (2010).

PA also anchors the cumulative ln Z at $\ln Z(0) = N \ln 2$ (uniform prior over $2^N$ states), so the absolute free energy density $F(\beta)/N = -\ln Z(\beta)/(\beta N)$ is comparable across runs and against the literature.

In [ ]:
betas = np.asarray(res_pa.history["beta"])
fdens = np.asarray(res_pa.history["free_energy_density"])
logz = np.asarray(res_pa.history["log_z"])

fig, axs = plt.subplots(1, 2, figsize=(11, 4), facecolor="white")
axs[0].plot(betas, fdens, marker="o", color="#C8005A", lw=2)
axs[0].set_xlabel("β")
axs[0].set_ylabel("F(β) / N")
axs[0].set_title("PA free-energy density estimate")
axs[0].grid(ls="--", alpha=0.5)
axs[0].set_xscale("log")

axs[1].plot(betas, logz, marker="o", color="#0050C8", lw=2)
axs[1].set_xlabel("β")
axs[1].set_ylabel("ln Z(β)")
axs[1].set_title("PA cumulative ln Z")
axs[1].grid(ls="--", alpha=0.5)
axs[1].set_xscale("log")
plt.tight_layout()
plt.show()

print(
    f"absolute F/N at β={BETA_END}: {res_pa.free_energy_density:.6f}\n"
    f"absolute ln Z at β={BETA_END}: {res_pa.log_z:.6f}"
)

### Sanity check: closed-form free energy on a 1-spin Ising

For $H = -h s$ with $s \in \{-1, +1\}$, the partition function is $Z(\beta) = 2\cosh(\beta h)$ and the free energy density is $F(\beta)/N = -\beta^{-1}\ln(2\cosh \beta h)$. PA should reproduce this within ~1 % when we use a large population and a long anneal.

In [ ]:
h = 0.7
beta_end_check = 3.0
res_check = qqa.population_annealing(
    qqa.Ising1D(N=1, h=h),
    sol_size=4096,
    num_temps=80,
    sweeps_per_temp=4,
    beta_start=0.05,
    beta_end=beta_end_check,
    seed=11,
    verbose=False,
)
f_exact = -math.log(2.0 * math.cosh(beta_end_check * h)) / beta_end_check
print(
    f"PA   F/N = {res_check.free_energy_density:.6f}\n"
    f"exact F/N = {f_exact:.6f}\n"
    f"|err|     = {abs(res_check.free_energy_density - f_exact):.2e}"
)

## 7. Resampling family tree — three views of the population collapse

PA's `record_genealogy=True` saves the parent index of every replica at every step plus the running root-ancestor map. We use it to draw three complementary views of the population's evolutionary dynamics:

1. **Muller plot** *(top)* — stacked-area visualisation borrowed from evolutionary biology. Each colour = one initial **founder**; band thickness = fraction of the population descended from that founder at that β. **Bands appearing, expanding, and going extinct = clonal interference and selective sweeps in real time.** Founders ordered by founder index for stable colours across runs.
2. **Sorted ancestry matrix** *(middle)* — at every β, the surviving replicas are sorted by founder so the population looks like clean horizontal "clades". A clade that pinches off = that lineage went extinct; a clade that widens = it took over.
3. **Survivor curve** *(bottom)* — number of distinct surviving founders vs β. The classic PA collapse curve. Crossing $R / e$ marks effective bottlenecks; reaching ~1 means the population has lost almost all genetic diversity (warning sign — increase R or refine the β grid).

In [ ]:
parents = res_pa.genealogy["parents"]
betas_g = np.asarray(res_pa.genealogy["betas"])
R = res_pa.final_x.shape[0]
T = len(parents)

# Reconstruct each replica's founder index at every step by composing
# the parent maps. ``mat[t, r]`` = founder of slot r at step t.
current = np.arange(R)
anc_through_time = [current.copy()]
for t in range(T):
    current = current[parents[t].cpu().numpy()]
    anc_through_time.append(current.copy())
mat = np.stack(anc_through_time, axis=0)  # (T+1, R)
n_surv = np.array([len(np.unique(mat[t])) for t in range(T + 1)])

# Build per-step founder shares. ``shares[t, k]`` = fraction of the
# population at step t descended from founder ``founder_ids[k]``.
founder_ids = np.arange(R)
shares = np.zeros((T + 1, R), dtype=float)
for t in range(T + 1):
    counts = np.bincount(mat[t], minlength=R)
    shares[t] = counts / R

# x-axis = anneal progress in β. The initial population is anchored just
# below β_min on the log scale so the very first slice is visible.
betas_x = np.concatenate([[0.5 * float(betas_g.min())], betas_g])

# Stable, perceptually-uniform palette indexed by founder id.
cmap = plt.get_cmap("turbo", R)
colors = [cmap(int(k)) for k in founder_ids]

fig = plt.figure(figsize=(10, 9.5), facecolor="white")
gs = fig.add_gridspec(3, 1, height_ratios=[2.4, 2.0, 1.0], hspace=0.42)

# ---- (1) Muller plot: stacked-area founder shares ---------------------
ax_m = fig.add_subplot(gs[0])
ax_m.stackplot(
    betas_x,
    shares.T,
    colors=colors,
    linewidth=0.0,
    antialiased=True,
)
ax_m.set_xscale("log")
ax_m.set_xlim(betas_x.min(), betas_x.max())
ax_m.set_ylim(0.0, 1.0)
ax_m.set_xlabel("β (log scale)")
ax_m.set_ylabel("Population share")
ax_m.set_title(f"PA Muller plot — founder shares vs β (R = {R})")
ax_m.set_yticks([0.0, 0.25, 0.5, 0.75, 1.0])
ax_m.grid(False)

# ---- (2) Sorted ancestry matrix --------------------------------------
ax_s = fig.add_subplot(gs[1])
sorted_mat = np.sort(mat, axis=1)
ax_s.imshow(
    sorted_mat.T,
    aspect="auto",
    cmap="turbo",
    vmin=0,
    vmax=R - 1,
    interpolation="nearest",
    extent=(0, T, R, 0),
)
ax_s.set_xlabel("Temperature step")
ax_s.set_ylabel("Replica (sorted by founder)")
ax_s.set_title("Sorted ancestry matrix — clades widen / pinch off as lineages take over / die")

# ---- (3) Surviving founders vs β -------------------------------------
ax_n = fig.add_subplot(gs[2])
ax_n.plot(betas_x, n_surv, marker="o", color="#C8005A", lw=2)
ax_n.axhline(R / math.e, color="#0050C8", ls="--", lw=1, label=f"R/e ≈ {R / math.e:.0f}")
ax_n.set_xscale("log")
ax_n.set_xlabel("β")
ax_n.set_ylabel("Distinct founders")
ax_n.set_title("Population collapse — surviving founders vs β")
ax_n.legend(loc="upper right")
ax_n.grid(ls="--", alpha=0.5)

plt.show()

## Summary

* **PQQA** is the gradient-based parallel solver — usually the fastest at finding the lowest-energy configuration on hard combinatorial problems (Ichikawa & Arai, ICLR 2025).
* **SA** is the textbook Markov-chain baseline.
* **PA** complements both by also providing a **representative equilibrium sample** (`final_x`), an **unbiased free-energy estimate** that automatically accounts for resampling, and a **resampling family tree** that exposes how the population collapses across the anneal — all three of which are essential for thermodynamic analysis and physical-statistics use cases.

**References**

* Hukushima, K. & Iba, Y. (2003). *Population annealing and its application to a spin glass*. AIP Conf. Proc. **690**.
* Machta, J. (2010). *Population annealing with weighted averages*. PRE **82**.
* Wang, W., Machta, J. & Katzgraber, H. G. (2015). *Comparing Monte Carlo methods for finding ground states of Ising spin glasses*. PRE **92**.
* Ichikawa, Y. & Arai, Y. (2025). *Optimization by Parallel Quasi-Quantum Annealing with Gradient-Based Sampling*. ICLR. arXiv:2409.02135